In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm

SEED = 42
SPLIT_ORDER = ["Train", "Validation", "Test"]

sns.set_theme(style="whitegrid")


In [ ]:
DATASET_ID = "Kwaai/IMDB_Sentiment"

raw_dataset = load_dataset(
    DATASET_ID,
    download_mode="force_redownload",
)
for split_name, dataset in raw_dataset.items():
    print(
        f"{split_name:<15}"
        f"{len(dataset):>8,} reviews | "
        f"columns: {dataset.column_names}"
    )

In [ ]:
train_valid_split = raw_dataset["train"].train_test_split(
    test_size=5000,
    seed=SEED,
    stratify_by_column="label",
)

train_dataset = train_valid_split["train"]
valid_dataset = train_valid_split["test"]
test_dataset = raw_dataset["test"]

unlabelled_dataset = raw_dataset["unsupervised"]

datasets = {
    "Train": train_dataset,
    "Validation": valid_dataset,
    "Test": test_dataset,
}
class_rows = []

for split_name, dataset in datasets.items():
    counts = np.bincount(
        dataset["label"],
        minlength=2,
    )

    class_rows.append({
        "Split": split_name,
        "Negative": int(counts[0]),
        "Positive": int(counts[1]),
        "Total": int(counts.sum()),
    })

class_counts = (
    pd.DataFrame(class_rows)
    .set_index("Split")
    .reindex(SPLIT_ORDER)
)

class_counts.loc["Overall"] = class_counts.sum()

display(class_counts)

In [ ]:
length_data = []
all_tokens = []

for split_name, dataset in datasets.items():
    for text in dataset["text"]:
        tokens = text.split()

        length_data.append([
            split_name,
            len(tokens),
        ])

        all_tokens.extend(tokens)

length_df = pd.DataFrame(
    length_data,
    columns=["Split", "Length"],
)

token_counts = Counter(all_tokens)

In [ ]:
display(
    length_df
    .groupby("Split")["Length"]
    .describe()
    .round(1)
)

print("Total reviews:", len(length_df))
print("Total tokens:", len(all_tokens))
print("Unique tokens:", len(token_counts))

In [ ]:
sns.histplot(
    data=length_df,
    x="Length",
    bins=60,
)

plt.xlim(0, length_df["Length"].quantile(0.99))
plt.title("Review Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
sns.boxplot(
    data=length_df,
    x="Split",
    y="Length",
    showfliers=False,
)

plt.title("Review Length Across Dataset Splits")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Words")
plt.show()

In [ ]:
frequencies = np.array(
    list(token_counts.values())
)

vocabulary_summary = pd.DataFrame({
    "Metric": [
        "Total token occurrences",
        "Unique tokens",
        "Tokens occurring once",
        "Tokens occurring at most five times",
        "Rare-token proportion",
    ],
    "Value": [
        len(all_tokens),
        len(token_counts),
        (frequencies == 1).sum(),
        (frequencies <= 5).sum(),
        (frequencies <= 5).mean(),
    ],
})

print(f"Total token occurrences: {len(all_tokens):,}")
print(f"Unique tokens: {len(token_counts):,}")
print(f"Tokens occurring once: {(frequencies == 1).sum():,}")
print(f"Tokens occurring at most five times: {(frequencies <= 5).sum():,}")
print(f"Rare-token proportion: {(frequencies <= 5).mean():.2%}")

In [ ]:
top_tokens = pd.DataFrame(
    token_counts.most_common(15),
    columns=["Token", "Frequency"],
)

display(top_tokens)

In [ ]:
sns.barplot(
    data=top_tokens,
    x="Frequency",
    y="Token",
)

plt.title("Most Frequent Raw Tokens")
plt.xlabel("Frequency")
plt.ylabel("Token")
plt.show()

In [ ]:
frequency_bands = pd.Series({
    "Once": (frequencies == 1).sum(),
    "2–5 times": (
        (frequencies >= 2)
        & (frequencies <= 5)
    ).sum(),
    "6–10 times": (
        (frequencies >= 6)
        & (frequencies <= 10)
    ).sum(),
    "11–100 times": (
        (frequencies >= 11)
        & (frequencies <= 100)
    ).sum(),
    "More than 100": (
        frequencies > 100
    ).sum(),
})

display(
    frequency_bands
    .rename("Number of unique tokens")
    .to_frame()
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.5),
)

top_tokens_plot = top_tokens.sort_values(
    "Frequency"
)

axes[0].barh(
    top_tokens_plot["Token"],
    top_tokens_plot["Frequency"],
    color="#4C78A8",
)

axes[0].set_title("Most Frequent Raw Tokens")
axes[0].set_xlabel("Frequency")
axes[0].set_ylabel("Token")

axes[1].bar(
    frequency_bands.index,
    frequency_bands.values,
    color="#B279A2",
)

axes[1].set_yscale("log")
axes[1].set_title("Raw Vocabulary Frequency Distribution")
axes[1].set_xlabel("Token Frequency")
axes[1].set_ylabel("Number of Unique Tokens (log scale)")
axes[1].tick_params(
    axis="x",
    rotation=25,
)

plt.tight_layout()
plt.show()

In [ ]:
import sys

sys.path.insert(0, 'BiLSTM')

from src.data.preprocess import load_text_column_from_parquet, normalize_text, tokenize_text, text_to_ids
from src.data.vocab import load_vocab

train_path = './BiLSTM/data/raw/train-00000-of-00001.parquet'
vocab_path = './BiLSTM/data/vocab.json'

raw_texts = load_text_column_from_parquet(train_path)
vocab = load_vocab(vocab_path)
raw_example = raw_texts[0]
normalized_example = normalize_text(raw_example, lowercase=True)
tokens = tokenize_text(normalized_example)
token_ids = text_to_ids(raw_example, vocab=vocab, lowercase=True)

summary_df = pd.DataFrame(
    {
        'item': ['vocab_size', 'raw_review_count', 'example_token_count'],
        'value': [len(vocab), len(raw_texts), len(tokens)],
    }
)
summary_df

print('RAW EXAMPLE (first 600 chars):')
print(raw_example[:600])
print('\nNORMALIZED EXAMPLE (first 600 chars):')
print(normalized_example[:600])
print('\nFIRST 40 TOKENS:')
print(tokens[:40])
print('\nFIRST 40 TOKEN IDS:')
print(token_ids[:40])

In [ ]:
# Key Qwen-specific preprocessing functions.
import re

def clean_review(text: str) -> str:
    text = re.sub(r"<br\s*/?>", " ", str(text), flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

def truncate_head_tail(text: str, tokenizer, max_tokens: int) -> str:
    text = clean_review(text)
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    if len(token_ids) <= max_tokens:
        return text
    head = max_tokens // 2
    tail = max_tokens - head
    kept_ids = token_ids[:head] + token_ids[-tail:]
    return tokenizer.decode(kept_ids, skip_special_tokens=True)

In [ ]:
feature_candidates = [
    {'name': 'Unigram only', 'ngram_range': (1, 1)},
    {'name': 'Unigram + consecutive bigram', 'ngram_range': (1, 2)},
]

vectorizer = CountVectorizer(
    max_features=20_000,
    ngram_range=selected_setting['ngram_range'],
    min_df=2,
    max_df=0.95,
)

model = MultinomialNB(alpha=1.0)

In [ ]:
import yaml
from pathlib import Path
config_path = Path('./BiLSTM/configs/baseline.yaml')
with config_path.open('r', encoding='utf-8') as file_handle:
    config = yaml.safe_load(file_handle)

rows = []
for section_name in ['model', 'data', 'vocab', 'training']:
    section = config.get(section_name, {})
    for key, value in section.items():
        rows.append({'section': section_name, 'parameter': key, 'value': value})

config_df = pd.DataFrame(rows)
config_df

In [ ]:
# Key prompt/generation protocol used by zero-shot and few-shot inference.
SYSTEM_PROMPT = (
    "Classify the overall sentiment of the IMDb movie review. "
    "Reply with exactly one word: positive or negative."
)
GENERATION_CONFIG = {
    "do_sample": False,
    "max_new_tokens": 4,
}
LABEL_TEXT_TO_ID = {"negative": 0, "positive": 1}
def parse_label(raw_output: str):
    match = re.search(r"\b(positive|negative)\b", raw_output.lower())
    return None if match is None else LABEL_TEXT_TO_ID[match.group(1)]

In [ ]:
# LoRA configuration used in the completed fine-tuning run.
from peft import LoraConfig

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
)

In [ ]:
overall_metrics = pd.DataFrame(
    {
        "Accuracy": [0.8289, 0.8492, 0.8863, 0.8968, 0.9514],
        "Precision": [0.8172, 0.8620, 0.8722, 0.8817, 0.9575],
        "Recall": [0.8474, 0.8314, 0.9053, 0.9165, 0.9448],
        "F1-score": [0.8320, 0.8464, 0.8884, 0.8988, 0.9511],
    },
    index=[
        "Logistic Regression",
        "Naive Bayes",
        "Embedding–TextCNN",
        "BiLSTM",
        "Qwen3-0.6B + LoRA",
    ],
)

# Transpose: group bars by Accuracy, Precision, Recall and F1
ax = overall_metrics.T.plot(
    kind="bar",
    figsize=(12, 5.5),
    width=0.82,
    color=[
        "#4C78A8",
        "#F58518",
        "#54A24B",
        "#E45756",
        "#B279A2",
    ],
)

ax.set_ylim(0.75, 1.00)
ax.set_title("Overall Test Performance by Metric", pad=48)
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=0)

ax.legend(
    title="Model",
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.01),
)

ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import json

metrics_path = Path('./BiLSTM/outputs/results/lstm_baseline_metrics.json')
with metrics_path.open('r', encoding='utf-8') as file_handle:
    metrics = json.load(file_handle)

metrics_df = pd.DataFrame(
    [{'metric': key, 'value': value} for key, value in metrics.items()]
)
metrics_df

In [ ]:
misclassified_path = Path('./BiLSTM/outputs/results/lstm_baseline_misclassified_examples.json')
with misclassified_path.open('r', encoding='utf-8') as file_handle:
    misclassified_examples = json.load(file_handle)

misclassified_df = pd.DataFrame(misclassified_examples)
misclassified_df['error_type'] = misclassified_df.apply(
    lambda row: 'false_positive' if row['true_label'] == 0 and row['predicted_label'] == 1 else 'false_negative',
    axis=1,
)

summary = misclassified_df.groupby('error_type').agg(
    count=('index', 'count'),
    mean_token_count=('token_count', 'mean'),
    median_token_count=('token_count', 'median'),
)
summary

sample_columns = ['error_type', 'true_label', 'predicted_label', 'token_count', 'text']
for error_type in ['false_positive', 'false_negative']:
    sample_df = misclassified_df.loc[misclassified_df['error_type'] == error_type, sample_columns].head(3).copy()
    if not sample_df.empty:
        sample_df['text'] = sample_df['text'].str.slice(0, 400) + '...'
        print(f'\n{error_type.upper()} EXAMPLES')
        display(sample_df)
